# Restaurant Rating Prediction - Exploratory Data Analysis (EDA)

This notebook performs a comprehensive exploratory data analysis on the Dataset.csv file.
We will cover:
1. Dataset overview
2. Missing value analysis
3. Duplicate analysis
4. Outlier detection
5. Correlation heatmaps
6. Cuisine popularity analysis
7. City-wise rating analysis
8. Online delivery vs rating analysis
9. Cost vs rating analysis
10. Business insights after each visualization

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# For better display in notebooks
%matplotlib inline

In [ ]:
# Load the dataset
df = pd.read_csv('../data/Dataset.csv')
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

# Show first 5 rows
df.head()

## 1. Dataset Overview

Let's get a quick overview of the dataset including columns, data types, and basic statistics.

In [ ]:
# Columns and data types
print("Columns and their data types:")
for col in df.columns:
    print(f"{col}: {df[col].dtype}")

In [ ]:
# Basic statistics for numerical columns
print("\nBasic statistics for numerical columns:")
display(df.describe())

## 2. Missing Value Analysis

We will check for missing values in the dataset and visualize them.

In [ ]:
# Missing values count
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_data = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_percent})

# Show only columns with missing values
missing_with_values = missing_data[missing_data['Missing Values'] > 0]
if missing_with_values.shape[0] > 0:
    print("Columns with missing values:")
    display(missing_with_values.sort_values(by='Percentage', ascending=False))
else:
    print("No missing values found.")

# Visualize missing values
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.tight_layout()
plt.show()

## Business Insight: Missing Values
if missing_with_values.shape[0] > 0:
    print("Notice: Some columns have missing values. We will need to handle these appropriately in preprocessing.")
else:
    print("Great! No missing values in the dataset.")

## 3. Duplicate Analysis

Check for duplicate rows in the dataset.

In [ ]:
# Check for duplicate rows
duplicate_rows = df.duplicated()
num_duplicates = duplicate_rows.sum()
print(f"Number of duplicate rows: {num_duplicates}")

if num_duplicates > 0:
    print("Duplicate rows:")
    display(df[duplicate_rows])
else:
    print("No duplicate rows found.")

## 4. Outlier Detection

We will detect outliers in numerical columns using the IQR method and visualize with box plots.

In [ ]:
# Select numerical columns for outlier detection
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numerical columns: {numerical_cols}")

# Function to detect outliers using IQR
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detect outliers for each numerical column
outlier_summary = []
for col in numerical_cols:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    outlier_summary.append({
        'Column': col,
        'Outlier Count': outliers.shape[0],
        'Lower Bound': lower,
        'Upper Bound': upper
    })

outlier_df = pd.DataFrame(outlier_summary)
print("Outlier summary:")
display(outlier_df)

# Visualize outliers with box plots
n_cols = 3
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
plt.figure(figsize=(5*n_cols, 4*n_rows))
for i, col in enumerate(numerical_cols):
    plt.subplot(n_rows, n_cols, i+1)
    sns.boxplot(y=df[col])
    plt.title(f'Box plot of {col}')
plt.tight_layout()
plt.show()

## Business Insight: Outliers
high_outlier_cols = outlier_df[outlier_df['Outlier Count'] > 0]
if not high_outlier_cols.empty:
    print("Notice: Several columns have outliers. These may need to be handled during preprocessing.")
    print("Columns with the most outliers:")
    print(high_outlier_cols.nlargest(3, 'Outlier Count')[['Column', 'Outlier Count']].to_string(index=False))
else:
    print("Good! No outliers detected in numerical columns.")

## 5. Correlation Heatmaps

We will compute the correlation matrix for numerical features and visualize it as a heatmap.

In [ ]:
# Compute correlation matrix
correlation_matrix = df[numerical_cols].corr()

# Plot heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

## Business Insight: Correlation
# Find highly correlated pairs (absolute correlation > 0.7)
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.7:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print("Highly correlated pairs (|correlation| > 0.7):")
    for pair in high_corr_pairs:
        print(f"{pair[0]} & {pair[1]}: {pair[2]:.4f}")
else:
    print("No highly correlated pairs found (threshold: 0.7).")

## 6. Cuisine Popularity Analysis

The 'Cuisines' column contains comma-separated values. We will analyze the popularity of different cuisines.

In [ ]:
# Split cuisines by comma and space, then explode and count
cuisines_split = df['Cuisines'].str.split(', ')
cuisines_exploded = cuisines_split.explode()
cuisine_popularity = cuisines_exploded.value_counts()

print(f"Total unique cuisines: {cuisine_popularity.shape[0]}")
print("\nTop 15 most common cuisines:")
display(cuisine_popularity.head(15))

In [ ]:
# Visualize top 15 cuisines
plt.figure(figsize=(12, 8))
top_15_cuisines = cuisine_popularity.head(15)
sns.barplot(x=top_15_cuisines.values, y=top_15_cuisines.index, palette='viridis')
plt.title('Top 15 Most Common Cuisines')
plt.xlabel('Count')
plt.ylabel('Cuisine')
plt.tight_layout()
plt.show()

## Business Insight: Cuisine Popularity
print("Insight: The most common cuisines are likely to be popular among customers and may influence ratings.")
print(f"The top cuisine is '{cuisine_popularity.index[0]}' with {cuisine_popularity.iloc[0]} occurrences.")

## 7. City-wise Rating Analysis

Analyze the average rating for each city to see which cities have the highest-rated restaurants.

In [ ]:
# City-wise average ratings
city_rating = df.groupby('City')['Aggregate rating'].agg(['mean', 'count']).rename(columns={'mean': 'Average Rating', 'count': 'Number of Restaurants'})
city_rating = city_rating.sort_values('Average Rating', ascending=False)

print("Top 10 Cities by Average Rating (with at least 5 restaurants):")
# Filter cities with at least 5 restaurants for reliability
city_rating_filtered = city_rating[city_rating['Number of Restaurants'] >= 5]
display(city_rating_filtered.head(10))

In [ ]:
# Visualize top 10 cities by average rating
plt.figure(figsize=(12, 8))
top_10_cities = city_rating_filtered.head(10)
sns.barplot(x=top_10_cities['Average Rating'], y=top_10_cities.index, palette='plasma')
plt.title('Top 10 Cities by Average Rating (min 5 restaurants)')
plt.xlabel('Average Rating')
plt.ylabel('City')
plt.tight_layout()
plt.show()

## Business Insight: City-wise Rating
print("Insight: Cities with higher average ratings may have better restaurant quality or different customer preferences.")
print(f"The top city is '{top_10_cities.index[0]}' with an average rating of {top_10_cities.iloc[0]['Average Rating']:.2f}.")

## 8. Online Delivery vs Rating Analysis

Check if restaurants with online delivery have higher ratings than those without.

In [ ]:
# Online delivery impact on rating
delivery_rating = df.groupby('Has Online delivery')['Aggregate rating'].agg(['mean', 'count']).rename(columns={'mean': 'Average Rating', 'count': 'Number of Restaurants'})
print("Average rating by online delivery availability:")
display(delivery_rating)

In [ ]:
# Visualize online delivery vs rating
plt.figure(figsize=(8, 6))
sns.boxplot(x='Has Online delivery', y='Aggregate rating', data=df, palette='Set2')
plt.title('Aggregate Rating by Online Delivery Availability')
plt.xlabel('Has Online Delivery')
plt.ylabel('Aggregate Rating')
plt.tight_layout()
plt.show()

## Business Insight: Online Delivery
yes_rating = delivery_rating.loc['Yes', 'Average Rating'] if 'Yes' in delivery_rating.index else 0
no_rating = delivery_rating.loc['No', 'Average Rating'] if 'No' in delivery_rating.index else 0
print(f"Restaurants with online delivery have an average rating of {yes_rating:.2f},")
print(f"while those without have an average rating of {no_rating:.2f}.")
if yes_rating > no_rating:
    print("Insight: Offering online delivery is associated with higher ratings.")
elif yes_rating < no_rating:
    print("Insight: Restaurants without online delivery have higher ratings, possibly due to focusing on dine-in experience.")
else:
    print("Insight: Online delivery availability does not seem to affect ratings.")

## 9. Cost vs Rating Analysis

Analyze the relationship between the average cost for two and the aggregate rating.

In [ ]:
# Cost vs rating scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Average Cost for two', y='Aggregate rating', data=df, alpha=0.6)
plt.title('Average Cost for Two vs Aggregate Rating')
plt.xlabel('Average Cost for Two')
plt.ylabel('Aggregate Rating')
plt.tight_layout()
plt.show()

# Calculate correlation
cost_rating_corr = df['Average Cost for two'].corr(df['Aggregate rating'])
print(f"Correlation between cost and rating: {cost_rating_corr:.4f}")

# Box plot of rating by price range (if available)
if 'Price range' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Price range', y='Aggregate rating', data=df, palette='viridis')
    plt.title('Aggregate Rating by Price Range')
    plt.xlabel('Price Range (1=lowest, 4=highest)')
    plt.ylabel('Aggregate Rating')
    plt.tight_layout()
    plt.show()
    
    # Average rating by price range
    price_range_rating = df.groupby('Price range')['Aggregate rating'].mean()
    print("\nAverage rating by price range:")
    display(price_range_rating)
    else:
    print("Price range column not found.")

## Business Insight: Cost vs Rating
print(f"The correlation between cost and rating is {cost_rating_corr:.4f}.")
if abs(cost_rating_corr) < 0.3:
    print("Insight: There is a weak correlation between cost and rating, suggesting that higher cost does not necessarily mean higher ratings.")
    elif cost_rating_corr > 0.3:
    print("Insight: There is a positive correlation between cost and rating, suggesting that higher-priced restaurants tend to have higher ratings.")
    else:
    print("Insight: There is a negative correlation between cost and rating, suggesting that higher-priced restaurants tend to have lower ratings (possibly due to value for money concerns).")

## Conclusion

We have completed the exploratory data analysis. Key findings:
1. Dataset shape and basic overview
2. Missing values and duplicates
3. Outliers in numerical features
4. Correlation between features
5. Cuisine popularity
6. City-wise rating differences
7. Impact of online delivery on ratings
8. Relationship between cost and rating

These insights will inform our feature engineering and modeling steps.